# 14o -- Edge Classifier Eval (leak-free MTMC eval of the learned Stage-4 gate)

CPU-only, self-contained MTMC eval for the Stage-4 learned edge classifier
(`docs/subagent-specs/edge-classifier-association.md` sections 5-7). Builds on
the 14n de-risk probe (PASSED: held-out hard-negative AUC +0.0735 over the cosine
threshold) and answers the real question: **does the learned gate move actual
MTMC IDF1 / id_switches off the 154 floor?**

Pipeline:
1. Clone `paper-tests` (reproduces 0.77936 + carries build_edge_pairs imports).
2. Inline the (uncommitted) refactored `build_edge_pairs.py` + new
   `edge_classifier.py` via base64; PATCH the cloned `pipeline.py` to insert the
   Stage-4 hook (assert the patch applied). **No git push needed.**
3. Assemble the 14e B1 stack (primary CLIP + DINOv2 tertiary; quaternary OFF).
4. Build labelled pairs per scene, train TWO LightGBM fold models
   (model_S02 on S02 pairs, model_S01 on S01 pairs).
5. **Drift gate**: edge_classifier OFF must reproduce **0.77936 / id_switches 154**.
6. **Leak-free eval**: apply model_S02 to S01 associations and model_S01 to S02
   associations; sweep blend_lambda x prob_threshold; report MTMC IDF1 +
   id_switches per config.
7. Final verdict table + `14o_edge_classifier_summary.json`.

Pre-registered bands: WIN >= 0.7820, MARGINAL >= 0.7810. KEY signal =
id_switches moving off 154.

## 1. Imports + paths

In [ ]:
import base64
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from datetime import datetime
from pathlib import Path

import numpy as np

WORK_DIR = Path('/kaggle/working')
PROJECT = WORK_DIR / 'gp'
INPUT_ROOT = Path('/kaggle/input')
ASSEMBLED_RUN = Path('/tmp/edge_clf_run')        # assembled stage1/stage2 run dir
DATA_OUT = WORK_DIR / 'outputs'
OUT_DIR = DATA_OUT / '14o_edge_classifier'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path('/tmp/edge_clf_models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python: {sys.version.split()[0]}')
print(f'Kaggle input exists: {INPUT_ROOT.exists()}')

## 2. Clone repo (paper-tests) + install CPU deps

In [ ]:
REPO_URL = 'https://github.com/MRKDaGods/gp.git'
REPO_BRANCH = 'paper-tests'   # reproduces 0.77936; carries scripts/build_edge_pairs.py imports

if not PROJECT.exists():
    print(f'Cloning {REPO_URL} ({REPO_BRANCH}) ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, str(PROJECT)])
else:
    print('Repo present; pulling latest ...')
    subprocess.check_call(['git', '-C', str(PROJECT), 'pull', '--ff-only'])

os.chdir(str(PROJECT))
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))


def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])


# Stage 3-5 need faiss + networkx + omegaconf + loguru; the classifier needs
# lightgbm + scikit-learn; build_edge_pairs writes parquet (pandas + pyarrow).
pip('numpy', 'scipy', 'pandas', 'pyarrow', 'faiss-cpu', 'omegaconf', 'loguru',
    'networkx>=3.1', 'lightgbm', 'scikit-learn', 'pyyaml', 'motmetrics')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], cwd=str(PROJECT))
print(f'Repo ready at {PROJECT}')

## 3. Inline uncommitted source + PATCH pipeline.py

`build_edge_pairs.py` (refactored to expose `PairFeatureBuilder`) and the new
`edge_classifier.py` are uncommitted -> absent from the clone. We write them
from base64, then insert the Stage-4 hook into the cloned `pipeline.py` so the
kernel runs WITHOUT a git push. Every step is asserted.

In [ ]:
# --- (a) Inline the refactored build_edge_pairs.py (PairFeatureBuilder source of truth). ---
_BEP_B64 = (
    "..."
    "IFN5c3RlbUV4aXQobWFpbigpKQo="
)
_bep_path = PROJECT / 'scripts' / 'build_edge_pairs.py'
_bep_path.write_bytes(base64.b64decode(_BEP_B64))
print(f'Wrote {_bep_path} ({_bep_path.stat().st_size} bytes)')

# --- (b) Inline the new edge_classifier.py. ---
_EC_B64 = (
    "..."
    "KQogICAgcmV0dXJuIG91dAo="
)
_ec_path = PROJECT / 'src' / 'stage4_association' / 'edge_classifier.py'
_ec_path.write_bytes(base64.b64decode(_EC_B64))
print(f'Wrote {_ec_path} ({_ec_path.stat().st_size} bytes)')

# --- (c) PATCH pipeline.py: insert the Stage-4 hook after the combined_sim anchor. ---
_HOOK_B64 = (
    "CiAgICAjIFN0ZXAgNS1FQzogTGVhcm5lZCBlZGdlIGNsYXNzaWZpZXIgLyByZS1yYW5rZXIgKGRlZmF1bHQgT0ZGKS4KICAgICMgUmVzY29yZXMgY29tYmlu"
    "ZWRfc2ltIHdpdGggYSBsZWFybmVkIHBlci1lZGdlIFAoc2FtZS12ZWhpY2xlKSBtb2RlbCBiZWZvcmUKICAgICMgYW55IHBvc3QtYWRqdXN0bWVudCBvciBn"
    "cmFwaCBzb2x2ZS4gYmxlbmRfbGFtYmRhPTAgKyBwcm9iX3RocmVzaG9sZDw9MCBpcyBhCiAgICAjIHByb3ZhYmxlIG5vLW9wIChyZXR1cm5zIGNvbWJpbmVk"
    "X3NpbSB1bmNoYW5nZWQpLiBTZWUKICAgICMgZG9jcy9zdWJhZ2VudC1zcGVjcy9lZGdlLWNsYXNzaWZpZXItYXNzb2NpYXRpb24ubWQgc2VjdGlvbnMgNS03"
    "LgogICAgZWRnZV9jbGZfcHJvYnM6IE9wdGlvbmFsW0RpY3RbVHVwbGVbaW50LCBpbnRdLCBmbG9hdF1dID0gTm9uZQogICAgZWNfY2ZnID0gc3RhZ2VfY2Zn"
    "LmdldCgiZWRnZV9jbGFzc2lmaWVyIiwge30pCiAgICBpZiBlY19jZmcuZ2V0KCJlbmFibGVkIiwgRmFsc2UpOgogICAgICAgIGZyb20gc3JjLnN0YWdlNF9h"
    "c3NvY2lhdGlvbi5lZGdlX2NsYXNzaWZpZXIgaW1wb3J0IHJlc2NvcmVfZWRnZXMKCiAgICAgICAgIyBjb3NfZnVzZWQgZnVzaW9uIHdlaWdodHMgbWlycm9y"
    "IHRoZSBzY29yZS1sZXZlbCBmdXNpb24gKFN0ZXAgM2IpOgogICAgICAgICMgdGVydGlhcnkgc3RyZWFtID09IERJTk92MiwgcXVhdGVybmFyeSBzdHJlYW0g"
    "PT0gUjUwLUlCTi4KICAgICAgICBlY19mdXNpb25fd2VpZ2h0cyA9ICgKICAgICAgICAgICAgcm91bmQoMS4wIC0gc2VjX3dlaWdodCAtIHRlcnRfd2VpZ2h0"
    "IC0gcXVhdF93ZWlnaHQsIDYpLAogICAgICAgICAgICB0ZXJ0X3dlaWdodCwKICAgICAgICAgICAgcXVhdF93ZWlnaHQsCiAgICAgICAgKQogICAgICAgICMg"
    "bWVhbiBjb25maWRlbmNlIHBlciB0cmFja2xldCBmcm9tIFN0YWdlLTEgKG1hdGNoZXMgYnVpbGRfZWRnZV9wYWlycykuCiAgICAgICAgZWNfdHJhY2tsZXRf"
    "bG9va3VwOiBEaWN0W1R1cGxlW3N0ciwgaW50XSwgVHJhY2tsZXRdID0ge30KICAgICAgICBmb3IgX2NhbV9pZCwgX3RyYWNrcyBpbiB0cmFja2xldHNfYnlf"
    "Y2FtZXJhLml0ZW1zKCk6CiAgICAgICAgICAgIGZvciBfdCBpbiBfdHJhY2tzOgogICAgICAgICAgICAgICAgZWNfdHJhY2tsZXRfbG9va3VwWyhfdC5jYW1l"
    "cmFfaWQsIF90LnRyYWNrX2lkKV0gPSBfdAogICAgICAgIHRyYWNrX2lkcyA9IFtmLnRyYWNrX2lkIGZvciBmIGluIGZlYXR1cmVzXQogICAgICAgIG1lYW5f"
    "Y29uZnMgPSBbCiAgICAgICAgICAgIGVjX3RyYWNrbGV0X2xvb2t1cFsoY2FtLCB0aWQpXS5tZWFuX2NvbmZpZGVuY2UKICAgICAgICAgICAgaWYgKGNhbSwg"
    "dGlkKSBpbiBlY190cmFja2xldF9sb29rdXAgZWxzZSAwLjAKICAgICAgICAgICAgZm9yIGNhbSwgdGlkIGluIHppcChjYW1lcmFfaWRzLCB0cmFja19pZHMp"
    "CiAgICAgICAgXQogICAgICAgIGVkZ2VfY2xmX3Byb2JzID0ge30KICAgICAgICBuX2JlZm9yZSA9IGxlbihjb21iaW5lZF9zaW0pCiAgICAgICAgY29tYmlu"
    "ZWRfc2ltID0gcmVzY29yZV9lZGdlcygKICAgICAgICAgICAgY29tYmluZWRfc2ltLAogICAgICAgICAgICBwcmltYXJ5PWVtYmVkZGluZ3MsCiAgICAgICAg"
    "ICAgIHRlcnRpYXJ5PXRlcnRfZW1iZWRkaW5ncywKICAgICAgICAgICAgcXVhdGVybmFyeT1xdWF0X2VtYmVkZGluZ3MsCiAgICAgICAgICAgIGNhbWVyYV9p"
    "ZHM9Y2FtZXJhX2lkcywKICAgICAgICAgICAgY2xhc3NfaWRzPWNsYXNzX2lkcywKICAgICAgICAgICAgdHJhY2tfaWRzPXRyYWNrX2lkcywKICAgICAgICAg"
    "ICAgc3RhcnRfdGltZXM9c3RhcnRfdGltZXMsCiAgICAgICAgICAgIGVuZF90aW1lcz1lbmRfdGltZXMsCiAgICAgICAgICAgIG51bV9mcmFtZXM9bnVtX2Zy"
    "YW1lcywKICAgICAgICAgICAgbWVhbl9jb25mcz1tZWFuX2NvbmZzLAogICAgICAgICAgICBzdF92YWxpZGF0b3I9c3RfdmFsaWRhdG9yLAogICAgICAgICAg"
    "ICBmdXNpb25fd2VpZ2h0cz1lY19mdXNpb25fd2VpZ2h0cywKICAgICAgICAgICAgZWNfY2ZnPWVjX2NmZywKICAgICAgICAgICAgZWRnZV9wcm9ic19vdXQ9"
    "ZWRnZV9jbGZfcHJvYnMsCiAgICAgICAgKQogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICBmIkVkZ2UgY2xhc3NpZmllciByZXNjb3JlZCB7bl9i"
    "ZWZvcmV9IGVkZ2VzIC0+IHtsZW4oY29tYmluZWRfc2ltKX0gIgogICAgICAgICAgICBmImtlcHQgKGZ1c2lvbl93ZWlnaHRzPXtlY19mdXNpb25fd2VpZ2h0"
    "c30pLiIKICAgICAgICApCg=="
)
HOOK_BLOCK = base64.b64decode(_HOOK_B64).decode('utf-8')
_pipe_path = PROJECT / 'src' / 'stage4_association' / 'pipeline.py'
_pipe_src = _pipe_path.read_text(encoding='utf-8')

ANCHOR = '    logger.info(f"Combined similarity pairs: {len(combined_sim)}")\n'
STEP5A = '    # Step 5a: Per-camera-pair similarity normalization.'
if 'Step 5-EC: Learned edge classifier' in _pipe_src:
    raise RuntimeError('pipeline.py already contains the edge-classifier hook (unexpected on paper-tests).')
if _pipe_src.count(ANCHOR) != 1:
    raise RuntimeError(f'Expected exactly 1 combined_sim anchor, found {_pipe_src.count(ANCHOR)}.')
if STEP5A not in _pipe_src:
    raise RuntimeError('Could not find the Step 5a marker to anchor the hook insertion.')

# Insert HOOK_BLOCK between the anchor line and the Step 5a comment.
_target = ANCHOR + '\n' + STEP5A
if _pipe_src.count(_target) != 1:
    raise RuntimeError('Anchor+Step5a target not found exactly once; pipeline.py layout changed.')
_replacement = ANCHOR + HOOK_BLOCK + '\n' + STEP5A
_pipe_src_new = _pipe_src.replace(_target, _replacement, 1)
if _pipe_src_new == _pipe_src:
    raise RuntimeError('Patch produced no change -- hook NOT inserted.')
_pipe_path.write_text(_pipe_src_new, encoding='utf-8')

# Assert the patch applied + the module imports cleanly with the hook present.
_check = _pipe_path.read_text(encoding='utf-8')
assert 'Step 5-EC: Learned edge classifier' in _check, 'hook marker missing after patch'
assert 'from src.stage4_association.edge_classifier import rescore_edges' in _check, 'hook import missing'

# Force a fresh import of the patched module (clear any cached import).
for _m in list(sys.modules):
    if _m.startswith('src.stage4_association') or _m == 'scripts.build_edge_pairs':
        del sys.modules[_m]
import importlib
import scripts.build_edge_pairs as _bep_mod
import src.stage4_association.edge_classifier as _ec_mod
import src.stage4_association.pipeline as _pipe_mod
assert hasattr(_bep_mod, 'PairFeatureBuilder'), 'PairFeatureBuilder missing from inlined build_edge_pairs'
assert hasattr(_ec_mod, 'rescore_edges'), 'rescore_edges missing from edge_classifier'
print('PATCH OK: hook inserted, modules import cleanly.')
print(f'  FEATURE_NAMES ({len(_bep_mod.FEATURE_NAMES)}): {_bep_mod.FEATURE_NAMES}')

## 4. Resolve 14h anchor, 14j quaternary, GT (from 14n)

kernel_sources mount under `/kaggle/input/notebooks/<owner>/<slug>/` (recurse
search as fallback). The 14h checkpoint supplies stage1 + primary CLIP +
DINOv2 tertiary; 14j supplies the R50-IBN quaternary (used only to BUILD the
classifier's `cos_r50ibn` feature -- the MTMC base stack keeps quaternary OFF).

In [ ]:
SOURCE_14H_OWNER_SLUG = 'yahiaakhalafallah/14h-robust-tracklet-pooling'
SOURCE_14J_OWNER_SLUG = 'yahiaakhalafallah/14j-r50-ibn-features'
SOURCE_14H_SLUG = SOURCE_14H_OWNER_SLUG.split('/', 1)[1]
SOURCE_14J_SLUG = SOURCE_14J_OWNER_SLUG.split('/', 1)[1]
EXPECTED_CAMS = ['S01_c001', 'S01_c002', 'S01_c003', 'S02_c006', 'S02_c007', 'S02_c008']
EXPECTED_TRACKLETS = 929


def find_input_dir(slug, owner_slug, hints=()):
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    owner, _, kernel = owner_slug.partition('/')
    nested = INPUT_ROOT / 'notebooks' / owner / kernel
    if nested.exists():
        return nested
    lowered_slug = slug.lower()
    lowered_hints = tuple(str(h).lower() for h in hints)
    for path in (list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []):
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(h in name for h in lowered_hints):
            return path
    return direct


def find_14h_checkpoint():
    source_dir = find_input_dir(SOURCE_14H_SLUG, SOURCE_14H_OWNER_SLUG, hints=('14h', 'robust', 'tracklet'))
    cp = source_dir / 'checkpoint.tar.gz'
    if cp.exists():
        print(f'14h input: {source_dir}')
        return cp
    visible = [str(p) for p in INPUT_ROOT.rglob('checkpoint.tar.gz')] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f'14h checkpoint.tar.gz not found for {SOURCE_14H_OWNER_SLUG}. Visible: {visible[:20]}')


checkpoint = find_14h_checkpoint()
EXTRACT_DIR = Path('/tmp/14h_checkpoint')
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)')
with tarfile.open(str(checkpoint), 'r:gz') as archive:
    archive.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / 'run_metadata.json', encoding='utf-8') as fh:
    previous_meta = json.load(fh)
SOURCE_14H_RUN_NAME = previous_meta['run_name']
SOURCE_14H_RUN_DIR = EXTRACT_DIR / SOURCE_14H_RUN_NAME
SOURCE_STAGE1_DIR = SOURCE_14H_RUN_DIR / 'stage1'
SOURCE_STAGE2_DIR = SOURCE_14H_RUN_DIR / 'stage2'
for required in [
    SOURCE_STAGE1_DIR,
    SOURCE_STAGE2_DIR / 'embeddings.npy',
    SOURCE_STAGE2_DIR / 'embeddings_tertiary.npy',
    SOURCE_STAGE2_DIR / 'hsv_features.npy',
    SOURCE_STAGE2_DIR / 'embedding_index.json',
]:
    if not required.exists():
        raise FileNotFoundError(required)
print(f'Loaded 14h run: {SOURCE_14H_RUN_NAME}')

In [ ]:
def find_quaternary_stage2_dir():
    source_dir = find_input_dir(SOURCE_14J_SLUG, SOURCE_14J_OWNER_SLUG, hints=('14j', 'r50', 'ibn'))
    candidates = [
        source_dir / 'outputs' / '14j_v4_features' / 'stage2',
        source_dir / '14j_v4_features' / 'stage2',
        source_dir / 'stage2',
    ]
    for cand in candidates:
        if (cand / 'embeddings_quaternary.npy').exists():
            print(f'14j quaternary input: {cand}')
            return cand
    matches = sorted(INPUT_ROOT.rglob('embeddings_quaternary.npy')) if INPUT_ROOT.exists() else []
    for m in matches:
        t = str(m).lower()
        if '14j' in t and ('r50' in t or 'ibn' in t or 'quaternary' in t):
            print(f'14j quaternary discovered: {m.parent}')
            return m.parent
    if matches:
        print(f'14j quaternary fallback: {matches[0].parent}')
        return matches[0].parent
    visible = [str(p) for p in INPUT_ROOT.rglob('*.npy')] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f'embeddings_quaternary.npy not found for {SOURCE_14J_OWNER_SLUG}. Visible npy: {visible[:30]}')


SOURCE_QUATERNARY_STAGE2_DIR = find_quaternary_stage2_dir()


def is_cityflow_gt_root(path):
    return path.exists() and all((path / cam / 'gt' / 'gt.txt').exists() for cam in EXPECTED_CAMS)


def find_cityflow_gt_root():
    candidates = [
        PROJECT / 'data' / 'raw' / 'cityflowv2',
        EXTRACT_DIR / 'gt_annotations',
        Path('/kaggle/input/data-aicity-2023-track-2'),
        Path('/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2'),
    ]
    for cand in candidates:
        if is_cityflow_gt_root(cand):
            return cand
    for gt_file in (INPUT_ROOT.rglob('gt.txt') if INPUT_ROOT.exists() else []):
        if gt_file.parent.name != 'gt' or gt_file.parent.parent.name not in EXPECTED_CAMS:
            continue
        cand = gt_file.parents[2]
        if is_cityflow_gt_root(cand):
            return cand
    visible = [str(p) for p in INPUT_ROOT.rglob('gt.txt')] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        'CityFlowV2 GT not found in <root>/<cam>/gt/gt.txt layout. '
        f'Expected {EXPECTED_CAMS}. Visible gt.txt: {visible[:20]}')


GT_DIR = find_cityflow_gt_root()
print(f'Ground truth root: {GT_DIR}')

## 5. Assemble a 2-STREAM run dir (stage1 + primary + DINOv2 tertiary)

CRITICAL train/inference-match decision: the MTMC base is the **clean 2-stream
14e B1 stack** (primary CLIP + DINOv2 tertiary; **quaternary R50-IBN OFF**). The
live pipeline therefore never loads the quaternary stream, so its hook produces
`cos_r50ibn = 0`. To keep the classifier's TRAINING features bit-identical to
what the pipeline feeds it at INFERENCE (the spec's hard 'train/infer
distribution match'), we build the labelled pairs on a **2-stream** run dir too
(`run.quaternary = None` -> `cos_r50ibn = 0` everywhere). The 14j quaternary is
still mounted + row-aligned-asserted (provenance), but intentionally NOT written
into the run -- including it would silently desync train vs infer on the
`cos_r50ibn / cos_min / cos_max / cos_std` features. `cos_std` over the two
present streams (primary, DINOv2) still carries the stream-disagreement signal.

In [ ]:
src_index = json.loads((SOURCE_STAGE2_DIR / 'embedding_index.json').read_text(encoding='utf-8'))
if len(src_index) != EXPECTED_TRACKLETS:
    raise RuntimeError(f'Expected {EXPECTED_TRACKLETS} rows, found {len(src_index)}')

# Provenance only: assert the 14j quaternary aligns to the 14h ordering, then
# DO NOT use it (2-stream base stack). This catches a stale/misaligned 14j mount.
quat_index = json.loads((SOURCE_QUATERNARY_STAGE2_DIR / 'embedding_index.json').read_text(encoding='utf-8'))
if src_index != quat_index:
    raise RuntimeError('14j quaternary embedding_index.json does not match 14h ordering')
quat_probe = np.load(SOURCE_QUATERNARY_STAGE2_DIR / 'embeddings_quaternary.npy').astype(np.float32)
if quat_probe.shape[0] != EXPECTED_TRACKLETS:
    raise RuntimeError(f'Unexpected quaternary shape: {quat_probe.shape}')
print(f'14j quaternary aligned (provenance only, NOT used): {quat_probe.shape}')

if ASSEMBLED_RUN.exists():
    shutil.rmtree(ASSEMBLED_RUN)
(ASSEMBLED_RUN / 'stage2').mkdir(parents=True, exist_ok=True)
shutil.copytree(SOURCE_STAGE1_DIR, ASSEMBLED_RUN / 'stage1')
# 2-stream stage2 ONLY: primary + DINOv2 tertiary + hsv + index. No quaternary.
for fname in ['embeddings.npy', 'embeddings_tertiary.npy', 'hsv_features.npy', 'embedding_index.json']:
    shutil.copy2(SOURCE_STAGE2_DIR / fname, ASSEMBLED_RUN / 'stage2' / fname)
assert not (ASSEMBLED_RUN / 'stage2' / 'embeddings_quaternary.npy').exists(), \
    'quaternary must be ABSENT from the 2-stream training run (would desync train vs infer)'

TERTIARY_PATH = ASSEMBLED_RUN / 'stage2' / 'embeddings_tertiary.npy'
print('Assembled 2-stream run dir stage2 contents:')
for p in sorted((ASSEMBLED_RUN / 'stage2').iterdir()):
    print(f'  stage2/{p.name}')
print(f'  stage1/: {len(list((ASSEMBLED_RUN / "stage1").glob("tracklets_*.json")))} tracklet files')

## 6. Sanity self-test the inlined build_edge_pairs.py

Exercises every feature-builder code path on tiny synthetic data -- catches an
env break before the real run.

In [ ]:
rc = subprocess.call([sys.executable, 'scripts/build_edge_pairs.py', '--self-test'])
print(f'self-test exit code: {rc}')
if rc != 0:
    raise RuntimeError('build_edge_pairs self-test failed -- fix env before the real run')

## 7. Build labelled pairs + train TWO leak-free fold models

`build_edge_pairs` produces per-pair features in the EXACT FIC+AQE space the
live gate uses. We use the **14e B1 base-stack fusion weights**
`(w_primary=0.475, w_tertiary=0.525, w_quaternary=0.0)` -- NOT the K7 registry
weights -- because the MTMC base stack is 2-stream, so `cos_fused` here equals
the appearance component the pipeline actually fuses at inference. We split by
scene and train:
* **model_S02** on S02 pairs only,
* **model_S01** on S01 pairs only.

At eval (Cell 9) model_S02 scores S01 associations and model_S01 scores S02
associations -- **no tracklet is ever scored by a model trained on its own
scene**. The fold dict is pickled to `models_by_train_scene` so
`edge_classifier.rescore_edges` applies the held-out model per scene
automatically (and asserts the mapping).

In [ ]:
import pickle
import importlib
import lightgbm as lgb

M = importlib.import_module('scripts.build_edge_pairs')
from src.stage4_association.spatial_temporal import SpatioTemporalValidator  # noqa

# Load the 2-stream run in the live FIC+AQE feature space (primary FIC+AQE;
# DINOv2 tertiary FIC-only; quaternary absent -> cos_r50ibn=0) -- identical
# transforms to the 14e B1 gate. AQE k=2 matches the base stack.
run = M.load_run(
    ASSEMBLED_RUN, GT_DIR, raw_cosines=False, fic_reg=M.DEFAULT_FIC_REG,
    fic_min_samples=M.DEFAULT_FIC_MIN_SAMPLES, aqe_k=M.DEFAULT_AQE_K,
    aqe_alpha=M.DEFAULT_AQE_ALPHA, top_k=M.DEFAULT_TOP_K,
)
if run.quaternary is not None:
    raise RuntimeError('Expected a 2-stream run (quaternary None); got a quaternary stream.')
# 14e B1 base-stack fusion weights -> cos_fused == pipeline appearance fusion.
FUSION_WEIGHTS = (0.475, 0.525, 0.0)
print(f'14e B1 base-stack fusion weights (NOT K7): {FUSION_WEIGHTS}')
st_validator = M._build_st_validator(M._load_camera_transitions())
cols = M.build_pairs(run, st_validator, fusion_weights=FUSION_WEIGHTS)

FEATURE_NAMES = list(M.FEATURE_NAMES)
CAT_FEATURES = list(M.CATEGORICAL_FEATURES)
X_all = np.column_stack([np.array(cols[name], dtype=np.float64) for name in FEATURE_NAMES])
y_all = np.array(cols['label'], dtype=np.int64)
scene_all = np.array(cols['scene'])
scenes_present = sorted(set(s for s in scene_all.tolist() if s))
print(f'Pairs total={len(y_all)} positives={int(y_all.sum())} scenes={scenes_present}')
if set(scenes_present) != {'S01', 'S02'}:
    raise RuntimeError(f'Expected exactly scenes S01+S02 for leak-free folds, got {scenes_present}')


def train_fold(train_scene):
    mask = scene_all == train_scene
    Xtr, ytr = X_all[mask], y_all[mask]
    if ytr.sum() == 0 or (ytr == 0).sum() == 0:
        raise RuntimeError(f'Degenerate labels for train scene {train_scene}: pos={int(ytr.sum())}')
    cat_idx = [FEATURE_NAMES.index(c) for c in CAT_FEATURES]
    mdl = M._train_lgbm(Xtr, ytr, cat_idx)
    print(f'  trained model_{train_scene}: n={len(ytr)} pos={int(ytr.sum())} neg={int((ytr==0).sum())}')
    return mdl


model_S02 = train_fold('S02')   # trained on S02 -> scores S01 associations
model_S01 = train_fold('S01')   # trained on S01 -> scores S02 associations

FOLD_PAYLOAD = {
    'feature_version': 1,
    'feature_names': FEATURE_NAMES,
    'fusion_weights': list(FUSION_WEIGHTS),
    'models_by_train_scene': {'S02': model_S02, 'S01': model_S01},
    'leak_free_mapping': {'S01_pairs': 'model_S02', 'S02_pairs': 'model_S01'},
}
MODEL_PATH = MODELS_DIR / 'edge_clf_lgbm_folds.pkl'
with MODEL_PATH.open('wb') as fh:
    pickle.dump(FOLD_PAYLOAD, fh)
print(f'Saved fold models -> {MODEL_PATH}')

# Assert the leak-free routing BEFORE any eval: scene X must be scored by the
# model whose train-scene != X.
from src.stage4_association.edge_classifier import EdgeClassifierModel
_ecm = EdgeClassifierModel(FOLD_PAYLOAD, MODEL_PATH)
assert _ecm.scene_to_model('S01') is model_S02, 'LEAK: S01 must be scored by model_S02'
assert _ecm.scene_to_model('S02') is model_S01, 'LEAK: S02 must be scored by model_S01'
print('LEAK-FREE MAPPING ASSERTED: S01->model_S02, S02->model_S01 (never train on the scored scene).')

## 8. Stage 3-5 driver (14e B1 base stack)

Mirrors the 14k eval driver's K0 config -- the clean 2-stream 14e B1 stack
(primary CLIP + DINOv2 tertiary, quaternary OFF, `aqe_k=2, sim_thr=0.48,
fic=0.5`) that reproduces **0.77936 / id_switches 154**. The only added knob is
the `edge_classifier.*` override block.

In [ ]:
from src.core.config import load_config, save_config
from src.core.data_models import TrackletFeatures
from src.core.io_utils import load_tracklets_by_camera
from src.core.logging_utils import setup_logging
from src.stage3_indexing import run_stage3
from src.stage4_association import run_stage4
from src.stage5_evaluation import run_stage5

RUN_NAME = f"run_14o_edge_clf_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
setup_logging(level='INFO', log_file=RUN_DIR / 'pipeline.log')
print(f'Run: {RUN_NAME}')

# --- 14e B1 base stack constants (K0 config from 14k) ---
BASE_PRIMARY_WEIGHT = 0.475
BASE_TERTIARY_WEIGHT = 0.525
SIM_THRESHOLD = 0.48
AQE_K = 2
FIC_REG = 0.5
SOLVER = 'cc'
ALGORITHM = 'conflict_free_cc'
LOUVAIN_RES = 0.70
APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)
BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38
INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30
MULTI_QUERY_WEIGHT = 0.0
MTMC_ONLY = False

K0_REPRO_TARGET = 0.77936
K0_REPRO_TOL = 0.001
K0_ID_SWITCH_TARGET = 154
WIN_THRESHOLD = 0.7820
MARGINAL_MIN = 0.7810


def load_metrics(report_path):
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding='utf-8'))
    details = payload.get('details', {}) or {}
    error_analysis = details.get('error_analysis', {}) or {}
    return {
        'mtmc_idf1': payload.get('mtmc_idf1', details.get('mtmc_idf1', payload.get('idf1'))),
        'trackeval_idf1': payload.get('idf1'),
        'idp': payload.get('idp', details.get('idp')),
        'idr': payload.get('idr', details.get('idr')),
        'mota': payload.get('mota'),
        'hota': payload.get('hota'),
        'id_switches': payload.get('id_switches'),
        'conflations': error_analysis.get('conflated_pred'),
        'fragmentations': error_analysis.get('fragmented_gt'),
        'num_pred_ids': payload.get('num_pred_ids', error_analysis.get('total_pred')),
    }


def build_features(stage2_dir):
    index_map = json.loads((stage2_dir / 'embedding_index.json').read_text(encoding='utf-8'))
    embeddings = np.load(stage2_dir / 'embeddings.npy').astype(np.float32)
    hsv_features = np.load(stage2_dir / 'hsv_features.npy').astype(np.float32)
    if embeddings.shape[0] != len(index_map) or hsv_features.shape[0] != len(index_map):
        raise ValueError(
            f'Stage2 row mismatch: embeddings={embeddings.shape}, hsv={hsv_features.shape}, index={len(index_map)}')
    return [
        TrackletFeatures(
            track_id=int(row['track_id']), camera_id=str(row['camera_id']),
            class_id=int(row['class_id']), embedding=embeddings[row_index],
            hsv_histogram=hsv_features[row_index], raw_embedding=None,
            multi_query_embeddings=None,
        )
        for row_index, row in enumerate(index_map)
    ]


def build_overrides(config, config_run_name):
    ec = config['edge_classifier']
    return [
        f'project.run_name={config_run_name}',
        f'project.output_dir={DATA_OUT}',
        'stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]',
        f'stage4.association.query_expansion.k={AQE_K}',
        'stage4.association.query_expansion.alpha=5.0',
        'stage4.association.query_expansion.dba=false',
        f'stage4.association.graph.similarity_threshold={SIM_THRESHOLD}',
        f'stage4.association.solver={SOLVER}',
        f'stage4.association.graph.algorithm={ALGORITHM}',
        f'stage4.association.graph.louvain_resolution={LOUVAIN_RES}',
        f'stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}',
        f'stage4.association.graph.max_component_size={MAX_COMP_SIZE}',
        f'stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}',
        f'stage4.association.weights.vehicle.hsv={HSV_WEIGHT}',
        f'stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}',
        'stage4.association.mutual_nn.top_k_per_query=20',
        'stage4.association.fic.enabled=true',
        f'stage4.association.fic.regularisation={FIC_REG}',
        'stage4.association.reranking.enabled=false',
        'stage4.association.camera_pair_norm.enabled=false',
        'stage4.association.fac.enabled=false',
        f'stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}',
        f'stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}',
        # --- 14e B1 base stack: quaternary OFF, DINOv2 tertiary ON ---
        'stage4.association.secondary_embeddings.path=',
        'stage4.association.secondary_embeddings.weight=0.0',
        f'stage4.association.tertiary_embeddings.path={TERTIARY_PATH}',
        f'stage4.association.tertiary_embeddings.weight={BASE_TERTIARY_WEIGHT}',
        'stage4.association.quaternary_embeddings.path=',
        'stage4.association.quaternary_embeddings.weight=0.0',
        'stage4.association.camera_bias.enabled=false',
        'stage4.association.zone_model.enabled=false',
        'stage4.association.hierarchical.enabled=false',
        f'stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}',
        f'stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}',
        f'stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}',
        'stage4.association.gallery_expansion.enabled=true',
        f'stage4.association.gallery_expansion.threshold={GALLERY_THRESH}',
        f'stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}',
        'stage4.association.weights.length_weight_power=0.3',
        'stage4.association.temporal_overlap.enabled=true',
        'stage4.association.temporal_overlap.bonus=0.05',
        'stage4.association.temporal_overlap.max_mean_time=5.0',
        # --- edge classifier overrides ---
        f'stage4.association.edge_classifier.enabled={str(ec["enabled"]).lower()}',
        f'stage4.association.edge_classifier.model_path={MODEL_PATH}',
        f'stage4.association.edge_classifier.mode={ec["mode"]}',
        f'stage4.association.edge_classifier.blend_lambda={ec["blend_lambda"]}',
        f'stage4.association.edge_classifier.prob_threshold={ec["prob_threshold"]}',
        # --- stage 5 ---
        f'stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}',
        'stage5.stationary_filter.enabled=true',
        'stage5.stationary_filter.min_displacement_px=150',
        'stage5.stationary_filter.max_mean_velocity_px=2.0',
        'stage5.min_submission_confidence=0.15',
        'stage5.cross_id_nms_iou=0.40',
        'stage5.min_trajectory_confidence=0.30',
        'stage5.min_trajectory_frames=40',
        'stage5.track_edge_trim.enabled=false',
        'stage5.track_smoothing.enabled=false',
        'stage5.gt_frame_clip=true',
        'stage5.gt_zone_filter=true',
        f'stage5.ground_truth_dir={GT_DIR}',
    ]


def run_config(config):
    config_id = config['config_id']
    config_dir = RUN_DIR / config_id
    config_dir.mkdir(parents=True, exist_ok=True)
    tracklets_by_camera = load_tracklets_by_camera(SOURCE_STAGE1_DIR)
    features = build_features(SOURCE_STAGE2_DIR)
    ec = config['edge_classifier']
    print('\n' + '=' * 80)
    print(f"Running {config_id}: edge_clf enabled={ec['enabled']} mode={ec['mode']} "
          f"lambda={ec['blend_lambda']} prob_thr={ec['prob_threshold']}")
    print('=' * 80)
    config_run_name = f'{RUN_NAME}_{config_id}'
    cfg = load_config(
        'configs/default.yaml', dataset_config='configs/datasets/cityflowv2.yaml',
        overrides=build_overrides(config, config_run_name),
    )
    save_config(cfg, config_dir / 'config.yaml')

    faiss_index, metadata_store = run_stage3(cfg, features, tracklets_by_camera, output_dir=config_dir / 'stage3')
    trajectories = run_stage4(cfg, faiss_index, metadata_store, features, tracklets_by_camera, output_dir=config_dir / 'stage4')
    run_stage5(cfg, trajectories, output_dir=config_dir / 'stage5')

    report_path = config_dir / 'stage5' / 'evaluation_report.json'
    metrics = load_metrics(report_path)
    pred_dir = config_dir / 'stage5' / 'predictions_mot'
    pred_files = sorted(pred_dir.glob('*.txt')) if pred_dir.exists() else []
    idf1_value = metrics.get('mtmc_idf1') or metrics.get('trackeval_idf1')
    if idf1_value is None:
        raise RuntimeError(f'IDF1 not found in {report_path}')
    if not pred_files:
        raise RuntimeError(f'No MOT prediction files written for {config_id}')
    row = {
        'config_id': config_id,
        'enabled': ec['enabled'], 'mode': ec['mode'],
        'blend_lambda': ec['blend_lambda'], 'prob_threshold': ec['prob_threshold'],
        'mtmc_idf1': metrics.get('mtmc_idf1'), 'trackeval_idf1': metrics.get('trackeval_idf1'),
        'idp': metrics.get('idp'), 'idr': metrics.get('idr'),
        'id_switches': metrics.get('id_switches'),
        'mota': metrics.get('mota'), 'hota': metrics.get('hota'),
        'conflations': metrics.get('conflations'), 'fragmentations': metrics.get('fragmentations'),
        'num_pred_ids': metrics.get('num_pred_ids'), 'num_trajectories': len(trajectories),
        'notes': config.get('notes', ''),
    }
    print(f"{config_id} MTMC IDF1: {float(idf1_value):.5f}  id_switches={row['id_switches']}")
    return row

## 9. DRIFT GATE + leak-free sweep

**Drift gate first**: edge_classifier OFF must reproduce 0.77936 / id_switches
154 (fail loud otherwise). Then enable the classifier and sweep
`blend_lambda in {0.0, 0.3, 0.5, 0.7}` x `prob_threshold in {0.0, 0.5, 0.6}`
(lambda=0, prob_thr=0 is the in-pipeline no-op sanity -- must ALSO equal 154).
Each enabled config applies model_S02 to S01 associations and model_S01 to S02
(leak-free). KEY signal: id_switches moving off 154.

In [ ]:
def ec_block(enabled, mode='blend', blend_lambda=0.0, prob_threshold=0.0):
    return {'enabled': enabled, 'mode': mode, 'blend_lambda': blend_lambda, 'prob_threshold': prob_threshold}


results = []
halt_reason = None

# --- (1) DRIFT GATE: classifier fully OFF ---
drift_cfg = {'config_id': 'D0_off', 'notes': 'drift gate: edge_classifier disabled = clean 14e B1',
             'edge_classifier': ec_block(False)}
drift_row = run_config(drift_cfg)
results.append(drift_row)
(OUT_DIR / '14o_partial_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

d_idf1 = float(drift_row['mtmc_idf1'])
d_idsw = drift_row.get('id_switches')
drift_ok = abs(d_idf1 - K0_REPRO_TARGET) <= K0_REPRO_TOL and d_idsw == K0_ID_SWITCH_TARGET
if not drift_ok:
    halt_reason = (f'DRIFT GATE FAILED: got idf1={d_idf1:.5f}, id_switches={d_idsw}; '
                   f'expected {K0_REPRO_TARGET:.5f} +/- {K0_REPRO_TOL} and id_switches={K0_ID_SWITCH_TARGET}')
    print(halt_reason)
    raise RuntimeError(halt_reason)
print(f'DRIFT GATE PASSED: idf1={d_idf1:.5f}, id_switches={d_idsw}')

# --- (2) lambda=0 no-op sanity (classifier ON, but provable in-pipeline no-op) ---
noop_cfg = {'config_id': 'N0_lambda0', 'notes': 'classifier ON but blend_lambda=0 prob_thr=0 -> in-pipeline no-op',
            'edge_classifier': ec_block(True, 'blend', 0.0, 0.0)}
noop_row = run_config(noop_cfg)
results.append(noop_row)
(OUT_DIR / '14o_partial_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
noop_ok = abs(float(noop_row['mtmc_idf1']) - K0_REPRO_TARGET) <= K0_REPRO_TOL and noop_row.get('id_switches') == K0_ID_SWITCH_TARGET
if not noop_ok:
    raise RuntimeError(f"lambda=0 NO-OP DRIFT: idf1={noop_row['mtmc_idf1']} id_switches={noop_row.get('id_switches')} "
                       f'(expected {K0_REPRO_TARGET} / {K0_ID_SWITCH_TARGET}). The hook is not a clean no-op.')
print(f"NO-OP SANITY PASSED: lambda=0 reproduces {noop_row['mtmc_idf1']:.5f} / id_switches {noop_row.get('id_switches')}")

# --- (3) LEAK-FREE SWEEP ---
BLEND_LAMBDAS = [0.0, 0.3, 0.5, 0.7]  # 0.0 + prob_thr>0 = pure removal gate (calibration-safe)
PROB_THRESHOLDS = [0.0, 0.4, 0.5, 0.6, 0.7]
for bl in BLEND_LAMBDAS:
    for pt in PROB_THRESHOLDS:
        cid = f'E_l{int(bl*100):03d}_p{int(pt*100):03d}'
        cfg_row = {'config_id': cid, 'notes': f'leak-free blend lambda={bl} prob_thr={pt}',
                   'edge_classifier': ec_block(True, 'blend', bl, pt)}
        row = run_config(cfg_row)
        results.append(row)
        (OUT_DIR / '14o_partial_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

print(f'\nCompleted {len(results)} configs.')

## 10. Verdict table + summary JSON

In [ ]:
BASE_IDF1 = K0_REPRO_TARGET
BASE_IDSW = K0_ID_SWITCH_TARGET

sweep_rows = [r for r in results if r['config_id'].startswith('E_')]
best = max(sweep_rows, key=lambda r: r['mtmc_idf1'] if r['mtmc_idf1'] is not None else -1.0) if sweep_rows else None
any_idsw_moved = any(r.get('id_switches') != BASE_IDSW for r in sweep_rows)
best_idf1 = float(best['mtmc_idf1']) if best else -1.0

if best_idf1 >= WIN_THRESHOLD:
    verdict = 'WIN'
elif best_idf1 >= MARGINAL_MIN:
    verdict = 'MARGINAL'
elif any_idsw_moved:
    verdict = 'NO-GO (id_switches moved but IDF1 below MARGINAL band)'
else:
    verdict = 'NO-GO (tie at 154 -- learned gate re-learned the threshold)'

print('=' * 92)
print('14o EDGE-CLASSIFIER LEAK-FREE MTMC EVAL -- VERDICT TABLE')
print('=' * 92)
print(f"{'config':<16}{'enabled':<9}{'lambda':<8}{'prob_thr':<10}{'MTMC_IDF1':<12}{'id_sw':<8}{'d_IDF1':<10}{'d_idsw':<8}")
print('-' * 92)
for r in results:
    idf1 = r['mtmc_idf1'] if r['mtmc_idf1'] is not None else float('nan')
    idsw = r.get('id_switches')
    d_idf1 = (idf1 - BASE_IDF1) if r['mtmc_idf1'] is not None else float('nan')
    d_idsw = (idsw - BASE_IDSW) if isinstance(idsw, int) else None
    print(f"{r['config_id']:<16}{str(r['enabled']):<9}{r['blend_lambda']:<8}{r['prob_threshold']:<10}"
          f"{idf1:<12.5f}{str(idsw):<8}{d_idf1:<+10.5f}{str(d_idsw):<8}")
print('-' * 92)
print(f'Base (drift) MTMC IDF1 = {BASE_IDF1:.5f}  id_switches = {BASE_IDSW}')
if best is not None:
    print(f"BEST sweep config = {best['config_id']}  MTMC IDF1 = {best_idf1:.5f}  "
          f"id_switches = {best.get('id_switches')}  (delta IDF1 = {best_idf1 - BASE_IDF1:+.5f})")
print(f'id_switches EVER moved off {BASE_IDSW}: {any_idsw_moved}')
print(f'Pre-registered bands: WIN >= {WIN_THRESHOLD}, MARGINAL >= {MARGINAL_MIN}')
print(f'VERDICT: {verdict}')
print('=' * 92)

summary = {
    'kernel': '14o_edge_classifier_eval',
    'base_stack': '14e B1 (primary CLIP + DINOv2 tertiary, quaternary OFF)',
    'fusion_weights': list(FUSION_WEIGHTS),
    'leak_free_protocol': {
        'description': 'two scene-disjoint LightGBM fold models; S01 associations scored by model_S02, '
                       'S02 associations scored by model_S01 (never train on the scored scene).',
        'S01_pairs_scored_by': 'model_S02', 'S02_pairs_scored_by': 'model_S01',
        'train_S01_n': int((scene_all == 'S01').sum()), 'train_S02_n': int((scene_all == 'S02').sum()),
        'train_S01_pos': int(y_all[scene_all == 'S01'].sum()), 'train_S02_pos': int(y_all[scene_all == 'S02'].sum()),
    },
    'drift_gate': {'target_idf1': K0_REPRO_TARGET, 'target_id_switches': K0_ID_SWITCH_TARGET,
                   'observed_idf1': float(drift_row['mtmc_idf1']), 'observed_id_switches': drift_row.get('id_switches'),
                   'passed': bool(drift_ok)},
    'bands': {'win': WIN_THRESHOLD, 'marginal': MARGINAL_MIN},
    'feature_names': FEATURE_NAMES,
    'best_sweep': best,
    'id_switches_moved': bool(any_idsw_moved),
    'verdict': verdict,
    'results': results,
}
summary_path = OUT_DIR / '14o_edge_classifier_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
print(f'Wrote {summary_path}')